<a href="https://colab.research.google.com/github/manaskanugo97-max/Healthcare-Lead-Gen-System/blob/main/app_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
%cd /content/drive/MyDrive/healthcare_lead_gen_project

/content/drive/MyDrive/healthcare_lead_gen_project


In [ ]:
import os
import pandas as pd
from config import FINAL_CSV

print("Dashboard input file:")
print(FINAL_CSV)

print("\nFinal CSV exists:")
print(os.path.exists(FINAL_CSV))

df = pd.read_csv(FINAL_CSV)

print("\nTotal rows:", len(df))
print("\nColumns:")
print(df.columns.tolist())

df.head()

Dashboard input file:
/content/drive/MyDrive/healthcare_lead_gen_project/data/healthcare_leads_final.csv

Final CSV exists:
True

Total rows: 100

Columns:
['Business Name', 'Industry Category', 'Business Description', 'Location', 'Google Maps Profile Link', 'OpenStreetMap Link', 'Website URL', 'Website Classification', 'Website Status Code', 'HTTPS Used', 'Contact Page Found', 'Services Page Found', 'Website Analysis Reason', 'Online Presence Type', 'Referral Platform Links', 'Phone Number', 'Email Address', 'Owner / Founder Name', 'LinkedIn Profile', 'Lead Score', 'Business Potential Category', 'Reason for Classification', 'Detailed Score Reason', 'Source', 'Scrape Status', 'Created Date']


,Business Name,Industry Category,Business Description,Location,Google Maps Profile Link,OpenStreetMap Link,Website URL,Website Classification,Website Status Code,HTTPS Used,...,Email Address,Owner / Founder Name,LinkedIn Profile,Lead Score,Business Potential Category,Reason for Classification,Detailed Score Reason,Source,Scrape Status,Created Date
0,Centre For Sight,clinic,Clinic listed on OpenStreetMap,"Block no. 124 Section AB, Scheme No. 54, Vijay...",https://www.google.com/maps/search/?api=1&quer...,https://www.openstreetmap.org/node/6187452264,NaN,Poor Website,NaN,NaN,...,NaN,NaN,NaN,65,High,Business has an online presence but lacks a st...,Website exists but digital quality is weak; Bu...,OpenStreetMap,Completed,2026-06-27
1,Dr. Vrinda Vashistha,clinic,Clinic listed on OpenStreetMap,"2-BB, Slice No.-5, Scheme No.-78, Vijay Nagar,...",https://www.google.com/maps/search/?api=1&quer...,https://www.openstreetmap.org/node/6187436057,NaN,Poor Website,NaN,NaN,...,NaN,NaN,NaN,65,High,Business has an online presence but lacks a st...,Website exists but digital quality is weak; Bu...,OpenStreetMap,Completed,2026-06-27
2,Well Care Pharmacy,pharmacy,Pharmacy listed on OpenStreetMap,"EB 303, Scheme No 94, Bombay Hospital Square, ...",https://www.google.com/maps/search/?api=1&quer...,https://www.openstreetmap.org/node/6187441510,NaN,Poor Website,NaN,NaN,...,NaN,NaN,NaN,65,High,Business has an online presence but lacks a st...,Website exists but digital quality is weak; Bu...,OpenStreetMap,Completed,2026-06-27
3,Rashmi Dental Clinic,dentist,Dentist listed on OpenStreetMap,"GH-28, Scheme No. 54, Vijay Nagar, Indore",https://www.google.com/maps/search/?api=1&quer...,https://www.openstreetmap.org/node/6187438667,NaN,Poor Website,NaN,NaN,...,NaN,NaN,NaN,65,High,Business has an online presence but lacks a st...,Website exists but digital quality is weak; Bu...,OpenStreetMap,Completed,2026-06-27
4,Dr. A. R. pawar,clinic,Clinic listed on OpenStreetMap,"AD 335, scheme 74c,vijay nagar, indore",https://www.google.com/maps/search/?api=1&quer...,https://www.openstreetmap.org/node/6187446196,NaN,Poor Website,NaN,NaN,...,NaN,NaN,NaN,65,High,Business has an online presence but lacks a st...,Website exists but digital quality is weak; Bu...,OpenStreetMap,Completed,2026-06-27


In [ ]:
!pip install streamlit pandas -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 81.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 118.9 MB/s eta 0:00:00


In [ ]:
%%writefile app.py

import streamlit as st
import pandas as pd
import os

from config import FINAL_CSV


st.set_page_config(
    page_title="Healthcare Lead Generation Dashboard",
    page_icon="🏥",
    layout="wide"
)


def load_data():
    if not os.path.exists(FINAL_CSV):
        st.error("Final CSV file not found. Please complete Step 6 first.")
        st.stop()

    df = pd.read_csv(FINAL_CSV)

    if "Lead Score" in df.columns:
        df["Lead Score"] = pd.to_numeric(df["Lead Score"], errors="coerce").fillna(0)

    return df


def safe_column(df, column_name):
    if column_name in df.columns:
        return df[column_name].fillna("").astype(str)
    return pd.Series([""] * len(df))


df = load_data()

st.title("🏥 Healthcare Lead Generation Dashboard")
st.write("This dashboard displays healthcare business leads collected from OpenStreetMap and enriched through online presence, website analysis, contact extraction, and lead scoring.")

st.divider()



total_leads = len(df)

high_potential = 0
medium_potential = 0
low_potential = 0
with_website = 0
without_website = 0

if "Business Potential Category" in df.columns:
    high_potential = len(df[df["Business Potential Category"] == "High"])
    medium_potential = len(df[df["Business Potential Category"] == "Medium"])
    low_potential = len(df[df["Business Potential Category"] == "Low"])

if "Website Classification" in df.columns:
    with_website = len(df[df["Website Classification"].isin(["Good Website", "Poor Website"])])
    without_website = len(df[df["Website Classification"] == "No Website"])

col1, col2, col3, col4, col5 = st.columns(5)

with col1:
    st.metric("Total Leads", total_leads)

with col2:
    st.metric("High Potential", high_potential)

with col3:
    st.metric("Medium Potential", medium_potential)

with col4:
    st.metric("With Website", with_website)

with col5:
    st.metric("No Website", without_website)

st.divider()



st.sidebar.header("Filters")

search_text = st.sidebar.text_input("Search Business Name / Location")

category_filter = "All"
if "Industry Category" in df.columns:
    categories = ["All"] + sorted(df["Industry Category"].dropna().astype(str).unique().tolist())
    category_filter = st.sidebar.selectbox("Industry Category", categories)

website_filter = "All"
if "Website Classification" in df.columns:
    website_options = ["All"] + sorted(df["Website Classification"].dropna().astype(str).unique().tolist())
    website_filter = st.sidebar.selectbox("Website Classification", website_options)

potential_filter = "All"
if "Business Potential Category" in df.columns:
    potential_options = ["All"] + sorted(df["Business Potential Category"].dropna().astype(str).unique().tolist())
    potential_filter = st.sidebar.selectbox("Business Potential", potential_options)

min_score = 0
max_score = 100

if "Lead Score" in df.columns:
    min_score, max_score = st.sidebar.slider(
        "Lead Score Range",
        min_value=0,
        max_value=100,
        value=(0, 100)
    )

sort_option = st.sidebar.selectbox(
    "Sort By",
    [
        "Lead Score High to Low",
        "Lead Score Low to High",
        "Business Name A-Z"
    ]
)



filtered_df = df.copy()

if search_text:
    search_text_lower = search_text.lower()

    business_series = safe_column(filtered_df, "Business Name").str.lower()
    location_series = safe_column(filtered_df, "Location").str.lower()

    filtered_df = filtered_df[
        business_series.str.contains(search_text_lower, na=False) |
        location_series.str.contains(search_text_lower, na=False)
    ]

if category_filter != "All" and "Industry Category" in filtered_df.columns:
    filtered_df = filtered_df[filtered_df["Industry Category"].astype(str) == category_filter]

if website_filter != "All" and "Website Classification" in filtered_df.columns:
    filtered_df = filtered_df[filtered_df["Website Classification"].astype(str) == website_filter]

if potential_filter != "All" and "Business Potential Category" in filtered_df.columns:
    filtered_df = filtered_df[filtered_df["Business Potential Category"].astype(str) == potential_filter]

if "Lead Score" in filtered_df.columns:
    filtered_df = filtered_df[
        (filtered_df["Lead Score"] >= min_score) &
        (filtered_df["Lead Score"] <= max_score)
    ]

if sort_option == "Lead Score High to Low" and "Lead Score" in filtered_df.columns:
    filtered_df = filtered_df.sort_values(by="Lead Score", ascending=False)

elif sort_option == "Lead Score Low to High" and "Lead Score" in filtered_df.columns:
    filtered_df = filtered_df.sort_values(by="Lead Score", ascending=True)

elif sort_option == "Business Name A-Z" and "Business Name" in filtered_df.columns:
    filtered_df = filtered_df.sort_values(by="Business Name", ascending=True)


# Dashboard Table


st.subheader("Lead Results")
st.write(f"Showing **{len(filtered_df)}** leads out of **{len(df)}** total leads.")

display_columns = [
    "Business Name",
    "Industry Category",
    "Location",
    "Website URL",
    "Website Classification",
    "Phone Number",
    "Email Address",
    "Lead Score",
    "Business Potential Category",
    "Reason for Classification"
]

available_display_columns = [col for col in display_columns if col in filtered_df.columns]

st.dataframe(
    filtered_df[available_display_columns],
    use_container_width=True,
    height=500
)



csv_data = filtered_df.to_csv(index=False).encode("utf-8-sig")

st.download_button(
    label="Download Filtered Leads as CSV",
    data=csv_data,
    file_name="filtered_healthcare_leads.csv",
    mime="text/csv"
)

st.divider()



st.subheader("Detailed Lead View")

if len(filtered_df) > 0 and "Business Name" in filtered_df.columns:
    selected_business = st.selectbox(
        "Select a business to view full details",
        filtered_df["Business Name"].astype(str).tolist()
    )

    selected_row = filtered_df[filtered_df["Business Name"].astype(str) == selected_business].iloc[0]

    col_a, col_b = st.columns(2)

    with col_a:
        st.write("### Business Information")
        st.write("**Business Name:**", selected_row.get("Business Name", ""))
        st.write("**Industry Category:**", selected_row.get("Industry Category", ""))
        st.write("**Description:**", selected_row.get("Business Description", ""))
        st.write("**Location:**", selected_row.get("Location", ""))
        st.write("**Google Maps Link:**", selected_row.get("Google Maps Profile Link", ""))

    with col_b:
        st.write("### Digital & Lead Intelligence")
        st.write("**Website URL:**", selected_row.get("Website URL", ""))
        st.write("**Website Classification:**", selected_row.get("Website Classification", ""))
        st.write("**Online Presence Type:**", selected_row.get("Online Presence Type", ""))
        st.write("**Lead Score:**", selected_row.get("Lead Score", ""))
        st.write("**Business Potential:**", selected_row.get("Business Potential Category", ""))
        st.write("**Reason:**", selected_row.get("Reason for Classification", ""))

    st.write("### Contact Information")
    st.write("**Phone Number:**", selected_row.get("Phone Number", ""))
    st.write("**Email Address:**", selected_row.get("Email Address", ""))
    st.write("**Owner / Founder Name:**", selected_row.get("Owner / Founder Name", ""))
    st.write("**LinkedIn Profile:**", selected_row.get("LinkedIn Profile", ""))

else:
    st.warning("No leads found for selected filters.")

Overwriting app.py


In [ ]:
!npm install -g localtunnel

⠙⠹⠸⠼⠴⠦⠧
changed 22 packages in 1s
⠧
⠧3 packages are looking for funding
⠧  run `npm fund` for details
⠧

In [ ]:
!streamlit run app.py --server.port 8501 & npx localtunnel --port 8501

⠙⠹⠸⠼⠴⠦

your url is: https://thirty-hornets-beg.loca.lt
2026-06-28 12:36:08.759 Uvicorn server started on 0.0.0.0:8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.125.117.142:8501

2026-06-28 12:36:42.038 Please replace `use_container_width` with `width`.

`use_container_width` will be removed after 2025-12-31.

For `use_container_width=True`, use `width='stretch'`. For `use_container_width=False`, use `width='content'`.
2026-06-28 12:37:02.576 Please replace `use_container_width` with `width`.

`use_container_width` will be removed after 2025-12-31.

For `use_container_width=True`, use `width='stretch'`. For `use_container_width=False`, use `width='content'`.
2026-06-28 12:37:25.292 Please replace `use_container_width` with `width`.

`use_container_width` will be removed after 2025-12-31.

For `use_container_width=True`, use `width='stretch'`. For `use_container_width=False

In [5]:
%cd /content/drive/MyDrive/healthcare_lead_gen_project

/content/drive/MyDrive/healthcare_lead_gen_project


In [6]:
!cp app.py app_backup.py
!cp config.py config_colab_backup.py

In [7]:
%%writefile config.py

import os

PROJECT_DIR = os.path.dirname(os.path.abspath(__file__))
DATA_DIR = os.path.join(PROJECT_DIR, "data")

RAW_OSM_CSV = os.path.join(DATA_DIR, "healthcare_osm_raw.csv")
ONLINE_PRESENCE_CSV = os.path.join(DATA_DIR, "healthcare_online_presence.csv")
WEBSITE_ANALYSIS_CSV = os.path.join(DATA_DIR, "healthcare_website_analysis.csv")
SCORED_LEADS_CSV = os.path.join(DATA_DIR, "healthcare_scored_leads.csv")
CONTACT_ENRICHED_CSV = os.path.join(DATA_DIR, "healthcare_contact_enriched.csv")
FINAL_CSV = os.path.join(DATA_DIR, "healthcare_leads_final.csv")

Overwriting config.py


In [8]:
from config import FINAL_CSV
import os

print(FINAL_CSV)
print(os.path.exists(FINAL_CSV))

/content/drive/MyDrive/healthcare_lead_gen_project/data/healthcare_leads_final.csv
True


In [10]:
!pip install streamlit pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 59.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 106.2 MB/s eta 0:00:00


In [11]:
!streamlit --version

Streamlit, version 1.58.0


In [1]:
%%writefile requirements.txt

streamlit
pandas

Writing requirements.txt


In [2]:
%%writefile requirements.txt

streamlit
pandas
requests
beautifulsoup4
overpy
ddgs
duckduckgo-search
tldextract
selenium

Overwriting requirements.txt


In [12]:
!streamlit run app.py --server.port 8501



2026-06-28 15:55:24.213 Uvicorn server started on 0.0.0.0:8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://35.227.77.223:8501

  Stopping...
  Stopping...
  Stopping...
  Stopping...
